In [ ]:
from qutip import destroy, tensor, qeye
from corral_crowding.detuning_fit import simulate_infidelity, fit_infidelity
import numpy as np


def dB_to_magnitude(A):
    return 10**(-1*A/10.)


def test_Hamiltonian_attenuations(detuning_list, lambdaq, eta, alpha, g3, qubit_attenuation_list, snail_attenuation_list):
    intra_prefactors = {
        "snail-qubit": 6 * eta * lambdaq * g3,
        "qubit-sub": 3 * eta**2 * lambdaq * g3,
        "qubit-qubit": 6 * eta * lambdaq**2 * g3,
    }

    inter_prefactors = {
        "snail-qubit (inter)": 6 * eta * lambdaq**3 * g3,
        "qubit-sub (inter)": 3 * eta**2 * lambdaq**3 * g3,
    }

    # Combine all prefactors
    prefactors = {**intra_prefactors, **inter_prefactors}

    # === Hilbert Space 1: Three Qubit System === #
    q = destroy(2)
    q1 = tensor(q, qeye(2), qeye(2))  # First qubit (Main interaction)
    q2 = tensor(qeye(2), q, qeye(2))  # Second qubit (Main interaction)
    q3 = tensor(qeye(2), qeye(2), q)  # Spectator qubit
    q1dag = q1.dag()
    q2dag = q2.dag()
    q3dag = q3.dag()

    # Intended gate (always between q1 and q2)
    intended_term_qubits = q1dag * q2 + q1 * q2dag
    ideal_gate_qubits = (-1.0j * (np.pi / 2) * intended_term_qubits).expm()

    # Spectator terms for qubit system
    spectator_ops_qubits = {
        "qubit-qubit": (q1dag * q3 + q1 * q3dag, ideal_gate_qubits)* qubit_attenuation_list[0],
        "qubit-sub": (q3dag + q3, ideal_gate_qubits) * qubit_attenuation_list[1],
        "qubit-sub (inter)": (q1dag + q1, ideal_gate_qubits) * qubit_attenuation_list[2],
    }

    # === Hilbert Space 2: Two Qubits + SNAIL System === #
    n_dim_snail = 8  # 8-Level SNAIL Mode
    qs = destroy(2)  # Qubit part of SNAIL system
    s = destroy(n_dim_snail)  # SNAIL oscillator
    qs1 = tensor(qs, qeye(2), qeye(n_dim_snail))  # First qubit
    qs2 = tensor(qeye(2), qs, qeye(n_dim_snail))  # Second qubit
    s1 = tensor(qeye(2), qeye(2), s)  # SNAIL mode
    qs1dag = qs1.dag()
    qs2dag = qs2.dag()
    s1dag = s1.dag()

    # Intended gate (always between qs1 and qs2)
    intended_term_snail = qs1dag * qs2 + qs1 * qs2dag
    ideal_gate_snail = (-1.0j * (np.pi / 2) * intended_term_snail).expm()

    # Spectator terms for SNAIL system
    spectator_ops_snail = {
        "snail-qubit": (qs1dag * s1 + qs1 * s1dag, ideal_gate_snail) * snail_attenuation_list[0],
        "snail-qubit (inter)": (qs1dag * s1 + qs1 * s1dag, ideal_gate_snail) * snail_attenuation_list[1],
    }

    # Compute infidelity curves and fit (a, b, c)
    infidelity_params = {}
    fidelity_results = {}

    # Compute for qubit-based spectators
    for key in spectator_ops_qubits:
        spectator_term, gate_target = spectator_ops_qubits[key]
        fidelity_results[key] = simulate_infidelity(
            detuning_list,
            intended_term_qubits,
            gate_target,
            prefactors[key],
            spectator_term,
        )
        infidelity_params[key] = fit_infidelity(detuning_list, fidelity_results[key])

    # Compute for SNAIL-based spectators
    for key in spectator_ops_snail:
        spectator_term, gate_target = spectator_ops_snail[key]
        fidelity_results[key] = simulate_infidelity(
            detuning_list,
            intended_term_snail,
            gate_target,
            prefactors[key],
            spectator_term,
        )
        infidelity_params[key] = fit_infidelity(detuning_list, fidelity_results[key])

    return infidelity_params, fidelity_results

In [ ]:
# list of attenuations dB: qubit-qubit, qubit-sub, qubit-sub(inter)
qubit_attenuation_list = np.array([0,10,0,0])
# list of attenuations dB: snail-qubit, snail-qubit(inter)
qubit_attenuation_list = np.array([0,10,0,0])